# Day 3 — Theorem proving with Lean 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/fmaiv/blob/main/notebooks/03_day3_theorem_proving.ipynb)

Builds the Day-3 Lean project [`day03/examples/CounterDemo/`](https://github.com/ttj/fmaiv/tree/main/day03/examples/CounterDemo) — the same `lake build` CI runs. The project is **Mathlib-free**, so once the Lean toolchain is present `lake build` finishes in seconds (no multi-gigabyte download). Preinstalled in Codespaces/the course image; on Colab the first build installs the pinned toolchain (a few minutes).

## Setup

In [ ]:
# --- Setup: find the course repo (clone it on Colab), define run helpers ------
# Idempotent: in GitHub Codespaces / the course image the repo and tools are
# already present, so the installs in the next cell are skipped. On Google Colab
# this clones the repo once. Re-running is safe.
import os, sys, re, subprocess, shutil, pathlib

def sh(cmd):
    """Run a shell command, streaming output; raise on failure."""
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

def run(cmd, expect=None):
    """Run a command, show its output, and (optionally) assert a verdict regex
    appears -- so this notebook self-checks exactly like CI (check_examples.sh)."""
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or '') + (r.stderr or '')
    print(out.rstrip())
    if expect is not None:
        assert re.search(expect, out), f'FAILED: expected /{expect}/ in output'
        print(f'  [ok] matched /{expect}/')
    elif r.returncode != 0:
        raise RuntimeError(f'command exited {r.returncode}')
    return out

def find_repo_root(marker='day01/examples'):
    for d in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
        if (d / marker).is_dir():
            return d
    return None

REPO = find_repo_root()
if REPO is None:                       # Colab: no repo on disk -> clone it once
    if not pathlib.Path('fmaiv').exists():
        sh('git clone --depth 1 https://github.com/ttj/fmaiv')
    REPO = pathlib.Path('fmaiv').resolve()
os.chdir(REPO)
assert (REPO / 'day01' / 'examples').is_dir(), 'unexpected repo layout'
print('Course repo:', REPO)

In [ ]:
# Lean 4 via elan. Preinstalled in Codespaces/the course image. On Colab we
# install elan + the toolchain pinned in the project's lean-toolchain file.
if not shutil.which('lake'):
    tc = (REPO / 'day03/examples/CounterDemo/lean-toolchain').read_text().strip()
    print('installing elan + toolchain', tc, '(a few minutes on first run)...')
    sh('curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -o /tmp/elan.sh '
       f'&& sh /tmp/elan.sh -y --default-toolchain {tc}')
    os.environ['PATH'] = os.path.expanduser('~/.elan/bin') + os.pathsep + os.environ['PATH']
print('lake:', shutil.which('lake') or 'NOT FOUND')
sh('lean --version')

## Build the project
Solution modules build clean and **`sorry`-free** (a complete proof); the `*Starter` modules are the exercises to fill in. `lake build` compiles the whole library — a green build means every proof checks.

In [ ]:
print('===== lake build (CounterDemo) =====')
run('cd day03/examples/CounterDemo && lake build')   # nonzero exit -> fails the cell (and CI)

Build a few individual modules — the standalone slide snippets (`SlideExamples` — the from-scratch `zero_add` and Gauss-sum `gauss` proofs walked through on the Day-3 induction slides), the discrete-math primer (a gentle Lean intro), and the counter invariant proof:

In [ ]:
for mod in ['CounterDemo.SlideExamples', 'CounterDemo.DiscreteMath', 'CounterDemo.Counter']:
    print(f'\n===== lake build {mod} =====')
    run(f'cd day03/examples/CounterDemo && lake build {mod}')

### Next steps
- **Live demo from the deck:** open `day03/examples/CounterDemo/CounterDemo/SlideExamples.lean` — the from-scratch `zero_add` and the Gauss-sum `gauss` proofs walked through on the Day-3 induction slides, with the goal state and induction hypothesis visible in the Lean InfoView as you step through the tactics.
- **Try it yourself:** open `day03/examples/CounterDemo/CounterDemo/CounterStarter.lean` (or `DiscreteMathStarter.lean`) in the Lean editor and replace each `sorry`; re-run the build.
- **Translate a Day-2 model into a Lean exercise:** `bash scripts/smv2lean/to_lean.sh day02/examples/mutex.smv` (worked examples live under `CounterDemo/NuXMV/`).
- **No install, in the browser:** the [Lean web editor](https://live.lean-lang.org/) and the [Natural Number Game](https://adam.math.hhu.de/).
- **Slides:** [Day 3 — Theorem proving](https://ttj.github.io/fmaiv/day03.html). **Next:** `04_day4_program_verif.ipynb`.